In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [2]:
BASE_DIR = Path(
    r"C:\Users\campus4D055\Desktop\통계프로젝트결측치파이썬보간계산이야호"
)

df = pd.read_pickle(
    BASE_DIR / "im_analysis_v1.pkl"
)

print("행 × 열:", df.shape)
print("법인 수:", df["법인ID"].nunique())
print("기준월 범위:", df["기준월"].min(), "~", df["기준월"].max())

행 × 열: (385756, 29)
법인 수: 15473
기준월 범위: 2023-01-01 00:00:00 ~ 2025-12-01 00:00:00


In [3]:
print(df.columns.tolist())

['법인ID', '기준월', '업종_대분류', '업종_중분류', '사업장_시도', '사업장_시군구', '예금잔액합계', '투자상품잔액', '운전자금대출잔액', '시설자금대출잔액', '카드사용액', '거래금액합계', '요구불입금금액', '요구불출금금액', '원본결측월', '결측구간번호', '결측개월수', '원본결측유형', '보간된변수수', '보간발생', '3개월이상결측구간', '예금잔액합계_보간여부', '투자상품잔액_보간여부', '운전자금대출잔액_보간여부', '시설자금대출잔액_보간여부', '카드사용액_보간여부', '거래금액합계_보간여부', '요구불입금금액_보간여부', '요구불출금금액_보간여부']


In [4]:
[c for c in df.columns if "보간여부" in c]

['예금잔액합계_보간여부',
 '투자상품잔액_보간여부',
 '운전자금대출잔액_보간여부',
 '시설자금대출잔액_보간여부',
 '카드사용액_보간여부',
 '거래금액합계_보간여부',
 '요구불입금금액_보간여부',
 '요구불출금금액_보간여부']

In [6]:
# ============================================================
# 09-0. 분석 대상 컬럼 확인
# ============================================================

TARGET_COLS = [
    "예금잔액합계",
    "투자상품잔액",
    "운전자금대출잔액",
    "시설자금대출잔액",
    "카드사용액",
    "거래금액합계",
    "요구불출금금액"
]

TARGET_FLAGS = [
    f"{c}_보간여부"
    for c in TARGET_COLS
]

print("분석 변수:")
for c in TARGET_COLS:
    print(c, "->", c in df.columns)

print("\n보간여부 변수:")
for c in TARGET_FLAGS:
    print(c, "->", c in df.columns)

분석 변수:
예금잔액합계 -> True
투자상품잔액 -> True
운전자금대출잔액 -> True
시설자금대출잔액 -> True
카드사용액 -> True
거래금액합계 -> True
요구불출금금액 -> True

보간여부 변수:
예금잔액합계_보간여부 -> True
투자상품잔액_보간여부 -> True
운전자금대출잔액_보간여부 -> True
시설자금대출잔액_보간여부 -> True
카드사용액_보간여부 -> True
거래금액합계_보간여부 -> True
요구불출금금액_보간여부 -> True


In [7]:
# ============================================================
# 09-A. 7개 핵심변수 공통 계절성 일괄 스크리닝
#
# 대상:
# 1) 예금잔액합계
# 2) 투자상품잔액
# 3) 운전자금대출잔액
# 4) 시설자금대출잔액
# 5) 카드사용액
# 6) 거래금액합계
# 7) 요구불출금금액
#
# 한 번에 수행:
# 1) 실제 관측값만 구성
# 2) 기업×연도 수준 제거
# 3) 2023/24/25 월효과
# 4) 3년 방향 반복성
# 5) Leave-One-Year-Out
# 6) 계절지수 후보
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 0. 분석용 기본 데이터
# ------------------------------------------------------------

work = df.copy()

if "기준월" in work.columns:

    work["기준월"] = pd.to_datetime(
        work["기준월"]
    )

elif "기준년월" in work.columns:

    work["기준월"] = pd.to_datetime(
        work["기준년월"].astype(str),
        format="%Y%m"
    )

else:

    raise ValueError(
        "기준월 또는 기준년월 컬럼이 없습니다."
    )


# ------------------------------------------------------------
# 실제 관측값만 꺼내는 함수
#
# 해당 변수의 _보간여부=True이면 제외
# 원래 결측값도 그대로 제외
# ------------------------------------------------------------

def observed_col(data, col):

    if col not in data.columns:
        raise KeyError(
            f"{col} 컬럼이 없습니다."
        )

    s = pd.to_numeric(
        data[col],
        errors="coerce"
    )

    flag = f"{col}_보간여부"

    if flag in data.columns:

        s = s.mask(
            data[flag].fillna(False)
        )

    return s


# ============================================================
# 1. 7개 분석변수 연결
#
# 이미 전처리 과정에서 합산 완료된 변수이므로
# 원본 세부변수를 다시 합치지 않음
# ============================================================

METRICS = {
    "예금잔액합계": "예금잔액합계",
    "투자상품잔액": "투자상품잔액",
    "운전자금대출잔액": "운전자금대출잔액",
    "시설자금대출잔액": "시설자금대출잔액",
    "카드사용액": "카드사용액",
    "거래금액합계": "거래금액합계",
    "요구불출금금액": "요구불출금금액"
}


# 실제 관측값 전용 분석 컬럼 생성
for metric_name, source_col in METRICS.items():

    analysis_col = (
        f"분석_{metric_name}"
    )

    work[analysis_col] = observed_col(
        work,
        source_col
    )


# 이후 분석에서는 분석_ 컬럼 사용
METRICS = {
    metric_name:
        f"분석_{metric_name}"

    for metric_name
    in METRICS.keys()
}


print("=" * 100)
print("7개 분석변수 연결 확인")
print("=" * 100)

for metric_name, value_col in METRICS.items():

    print(
        f"{metric_name:<15}"
        f" <- {value_col}"
        f" / 관측값 "
        f"{work[value_col].notna().sum():,}건"
    )


# ============================================================
# 2. 공통 계절성 분석 함수
# ============================================================

MIN_MONTHS = 6


def analyze_common_seasonality(
    data,
    metric_name,
    value_col
):

    x = data[
        [
            "법인ID",
            "기준월",
            "업종_대분류",
            value_col
        ]
    ].copy()

    x = x.rename(
        columns={
            value_col: "값"
        }
    )


    # --------------------------------------------------------
    # 결측 제외
    # --------------------------------------------------------

    x = x[
        x["값"].notna()
    ].copy()


    # --------------------------------------------------------
    # 금액변수이므로 음수 확인 후 제외
    # --------------------------------------------------------

    negative_n = (
        x["값"] < 0
    ).sum()

    x = x[
        x["값"] >= 0
    ].copy()


    # --------------------------------------------------------
    # 연도 / 월
    # --------------------------------------------------------

    x["연도"] = (
        x["기준월"].dt.year
    )

    x["월"] = (
        x["기준월"].dt.month
    )


    # --------------------------------------------------------
    # 큰 기업 몇 곳의 영향을 줄이기 위해 log1p
    # --------------------------------------------------------

    x["log값"] = np.log1p(
        x["값"]
    )


    # ========================================================
    # 기업×연도 수준 제거
    # ========================================================

    fy = (
        x
        .groupby(
            ["법인ID", "연도"]
        )
        .agg(
            관측월수=(
                "월",
                "nunique"
            ),

            연간_log평균=(
                "log값",
                "mean"
            )
        )
        .reset_index()
    )


    # 해당 기업×연도에서
    # 실제 관측월 최소 6개월
    fy = fy[
        fy["관측월수"] >= MIN_MONTHS
    ].copy()


    x = x.merge(
        fy[
            [
                "법인ID",
                "연도",
                "연간_log평균"
            ]
        ],
        on=[
            "법인ID",
            "연도"
        ],
        how="inner"
    )


    x["월효과_log"] = (
        x["log값"]
        - x["연간_log평균"]
    )


    # ========================================================
    # 연도 × 월 공통효과
    # ========================================================

    ym = (
        x
        .groupby(
            [
                "연도",
                "월"
            ]
        )
        .agg(
            월효과_log=(
                "월효과_log",
                "median"
            ),

            기업수=(
                "법인ID",
                "nunique"
            )
        )
        .reset_index()
    )


    ym["효과_pct"] = (
        np.expm1(
            ym["월효과_log"]
        )
        * 100
    )


    # ========================================================
    # 월별 3년 반복성 + Leave-One-Year-Out
    # ========================================================

    rows = []

    years = [
        2023,
        2024,
        2025
    ]


    for month in range(
        1,
        13
    ):

        t = (
            ym[
                ym["월"] == month
            ]
            .set_index("연도")
        )


        # 3개 연도가 모두 있어야 비교
        if not all(
            y in t.index
            for y in years
        ):
            continue


        logs = np.array(
            [
                t.loc[
                    y,
                    "월효과_log"
                ]
                for y in years
            ]
        )


        pcts = (
            np.expm1(logs)
            * 100
        )


        # ----------------------------------------------------
        # 3개 연도를 동일하게 반영한 대표효과
        # ----------------------------------------------------

        rep_log = np.median(
            logs
        )

        rep_pct = (
            np.expm1(rep_log)
            * 100
        )


        # ----------------------------------------------------
        # 3년 방향 반복성
        # ----------------------------------------------------

        all_up = np.all(
            logs > 0
        )

        all_down = np.all(
            logs < 0
        )

        same_direction = (
            all_up
            or all_down
        )


        if all_up:

            direction = "상승"

        elif all_down:

            direction = "하락"

        else:

            direction = "불일치"


        # ====================================================
        # Leave-One-Year-Out
        # ====================================================

        before_abs = []
        after_abs = []

        improved = 0


        for test_year in years:

            train_years = [
                y for y in years
                if y != test_year
            ]


            # 다른 두 연도로 계절효과 추정
            train_log = np.median(
                [
                    t.loc[
                        y,
                        "월효과_log"
                    ]
                    for y in train_years
                ]
            )


            # 제외했던 연도의 실제 효과
            test_log = (
                t.loc[
                    test_year,
                    "월효과_log"
                ]
            )


            # 조정 전
            before = abs(
                np.expm1(
                    test_log
                )
                * 100
            )


            # 조정 후
            after = abs(
                np.expm1(
                    test_log
                    - train_log
                )
                * 100
            )


            before_abs.append(
                before
            )

            after_abs.append(
                after
            )


            if after < before:

                improved += 1


        before_mean = np.mean(
            before_abs
        )

        after_mean = np.mean(
            after_abs
        )


        if before_mean > 0:

            reduction = (
                1
                - after_mean
                / before_mean
            ) * 100

        else:

            reduction = np.nan


        # ====================================================
        # 자동 스크리닝 등급
        #
        # A
        # 3년 동일방향
        # 대표효과 절대값 >= 1%
        # LOO 2/3 이상 개선
        # 평균 절대효과 20% 이상 감소
        #
        # B
        # 동일방향 + 1% 이상 + LOO 2/3 이상
        # 개선폭은 0~20%
        #
        # C
        # 방향은 반복되지만 고정지수 안정성 부족
        #
        # D
        # 3년 방향 자체가 불일치
        # ====================================================

        abs_effect = abs(
            rep_pct
        )


        if (
            same_direction
            and abs_effect >= 1
            and improved >= 2
            and reduction >= 20
        ):

            grade = (
                "A_조정후보"
            )


        elif (
            same_direction
            and abs_effect >= 1
            and improved >= 2
            and reduction > 0
        ):

            grade = (
                "B_약한후보"
            )


        elif same_direction:

            grade = (
                "C_반복하지만불안정"
            )


        else:

            grade = (
                "D_공통계절성약함"
            )


        rows.append(
            {
                "변수":
                    metric_name,

                "월":
                    month,

                "2023_pct":
                    pcts[0],

                "2024_pct":
                    pcts[1],

                "2025_pct":
                    pcts[2],

                "대표효과_pct":
                    rep_pct,

                "방향":
                    direction,

                "LOO_개선연도수":
                    improved,

                "LOO_조정전절대효과":
                    before_mean,

                "LOO_조정후절대효과":
                    after_mean,

                "LOO_감소율_pct":
                    reduction,

                "계절지수후보":
                    np.exp(
                        rep_log
                    ),

                "등급":
                    grade,

                "최소연도기업수":
                    int(
                        t[
                            "기업수"
                        ].min()
                    ),

                "음수제외건수":
                    int(
                        negative_n
                    )
            }
        )


    return (
        pd.DataFrame(
            rows
        ),
        ym,
        x
    )


# ============================================================
# 3. 7개 변수 일괄 실행
# ============================================================

all_screen = []

common_year_month_by_var = {}

analysis_long_by_var = {}


for (
    metric_name,
    value_col
) in METRICS.items():

    result, ym, long_data = (
        analyze_common_seasonality(
            work,
            metric_name,
            value_col
        )
    )


    all_screen.append(
        result
    )


    common_year_month_by_var[
        metric_name
    ] = ym


    analysis_long_by_var[
        metric_name
    ] = long_data


common_season_screen = (
    pd.concat(
        all_screen,
        ignore_index=True
    )
)


# ============================================================
# 4. 변수별 요약
# ============================================================

def months_of_grade(
    g,
    grade
):

    m = g.loc[
        g["등급"] == grade,
        "월"
    ].tolist()


    if not m:

        return "-"


    return ", ".join(
        f"{x}월"
        for x in m
    )


overview_rows = []


for (
    metric,
    g
) in (
    common_season_screen
    .groupby("변수")
):

    overview_rows.append(
        {
            "변수":
                metric,

            "A_조정후보":
                months_of_grade(
                    g,
                    "A_조정후보"
                ),

            "B_약한후보":
                months_of_grade(
                    g,
                    "B_약한후보"
                ),

            "C_불안정":
                months_of_grade(
                    g,
                    "C_반복하지만불안정"
                )
        }
    )


variable_overview = pd.DataFrame(
    overview_rows
)


print(
    "\n"
    + "=" * 110
)

print(
    "7개 변수 계절성 스크리닝 요약"
)

print(
    "=" * 110
)


print(
    variable_overview
    .to_string(
        index=False
    )
)


# ============================================================
# 5. A/B/C 후보 상세
# ============================================================

candidate_table = (
    common_season_screen[
        common_season_screen[
            "등급"
        ]
        != "D_공통계절성약함"
    ]
    .sort_values(
        [
            "등급",
            "변수",
            "월"
        ]
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "반복 계절효과 후보 상세"
)

print(
    "=" * 110
)


print(
    candidate_table[
        [
            "변수",
            "월",
            "2023_pct",
            "2024_pct",
            "2025_pct",
            "대표효과_pct",
            "방향",
            "LOO_개선연도수",
            "LOO_감소율_pct",
            "계절지수후보",
            "등급"
        ]
    ]
    .round(3)
    .to_string(
        index=False
    )
)


# ============================================================
# 6. 실제 공통 계절지수 후보만 별도
# ============================================================

index_candidates = (
    common_season_screen[
        common_season_screen[
            "등급"
        ]
        == "A_조정후보"
    ][
        [
            "변수",
            "월",
            "대표효과_pct",
            "계절지수후보",
            "LOO_개선연도수",
            "LOO_감소율_pct"
        ]
    ]
    .sort_values(
        [
            "변수",
            "월"
        ]
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "A등급: 공통 계절지수 후보"
)

print(
    "=" * 110
)


if len(
    index_candidates
):

    print(
        index_candidates
        .round(4)
        .to_string(
            index=False
        )
    )

else:

    print(
        "A등급 후보 없음"
    )

7개 분석변수 연결 확인
예금잔액합계          <- 분석_예금잔액합계 / 관측값 365,988건
투자상품잔액          <- 분석_투자상품잔액 / 관측값 365,988건
운전자금대출잔액        <- 분석_운전자금대출잔액 / 관측값 365,988건
시설자금대출잔액        <- 분석_시설자금대출잔액 / 관측값 365,988건
카드사용액           <- 분석_카드사용액 / 관측값 365,988건
거래금액합계          <- 분석_거래금액합계 / 관측값 365,988건
요구불출금금액         <- 분석_요구불출금금액 / 관측값 365,988건

7개 변수 계절성 스크리닝 요약
      변수               A_조정후보 B_약한후보      C_불안정
  거래금액합계 1월, 2월, 8월, 10월, 11월      - 3월, 6월, 9월
시설자금대출잔액                    -      -          -
  예금잔액합계             10월, 11월      -          -
 요구불출금금액 1월, 2월, 8월, 10월, 11월      -     3월, 6월
운전자금대출잔액                    -      -          -
   카드사용액                    -      -          -
  투자상품잔액                    -      -          -

반복 계절효과 후보 상세
     변수  월  2023_pct  2024_pct  2025_pct  대표효과_pct 방향  LOO_개선연도수  LOO_감소율_pct  계절지수후보         등급
 거래금액합계  1     3.221     5.691     2.704     3.221 상승          3       54.815   1.032     A_조정후보
 거래금액합계  2    -6.134    -1.419    -3.377    -3.377 하락         

In [8]:
# ============================================================
# 09-B. 7개 변수 업종별 추가 계절성 일괄 스크리닝
#
# A등급 공통 월효과가 있으면
# 먼저 해당 공통효과를 제거한 뒤
# 업종별 추가적인 계절효과가 남는지 검사
#
# 최소 업종 표본:
# 해당 업종 × 해당 월에서
# 각 연도 기업수 >= 50
# ============================================================

INDUSTRY_MIN_N = 50

industry_rows = []


for metric_name in METRICS.keys():

    x = (
        analysis_long_by_var[
            metric_name
        ]
        .copy()
    )

    ym = (
        common_year_month_by_var[
            metric_name
        ]
        .copy()
    )


    # ========================================================
    # 이 변수에서 A등급으로 판정된
    # 전체기업 공통 계절월
    # ========================================================

    common_A_months = set(
        common_season_screen.loc[
            (
                common_season_screen[
                    "변수"
                ]
                == metric_name
            )
            &
            (
                common_season_screen[
                    "등급"
                ]
                == "A_조정후보"
            ),
            "월"
        ]
    )


    # --------------------------------------------------------
    # 연도별 전체기업 공통 월효과
    # --------------------------------------------------------

    common_effect_map = (
        ym
        .set_index(
            [
                "연도",
                "월"
            ]
        )[
            "월효과_log"
        ]
        .to_dict()
    )


    # ========================================================
    # 업종 × 연도 × 월 효과
    # ========================================================

    ind_ym = (
        x
        .groupby(
            [
                "업종_대분류",
                "연도",
                "월"
            ]
        )
        .agg(
            업종효과_log=(
                "월효과_log",
                "median"
            ),

            기업수=(
                "법인ID",
                "nunique"
            )
        )
        .reset_index()
    )


    # ========================================================
    # 공통 A등급 효과 제거
    #
    # 공통계절월이면:
    # 업종 월효과 - 전체기업 공통효과
    #
    # 공통계절월이 아니면:
    # 업종 자체 월효과 그대로 사용
    # ========================================================

    def residual_effect(
        row
    ):

        if (
            row["월"]
            in common_A_months
        ):

            return (
                row[
                    "업종효과_log"
                ]
                -
                common_effect_map.get(
                    (
                        row["연도"],
                        row["월"]
                    ),
                    0
                )
            )

        return row[
            "업종효과_log"
        ]


    ind_ym[
        "잔여효과_log"
    ] = (
        ind_ym.apply(
            residual_effect,
            axis=1
        )
    )


    # ========================================================
    # 업종별 3년 반복성 + Leave-One-Year-Out
    # ========================================================

    years = [
        2023,
        2024,
        2025
    ]


    for (
        industry,
        month
    ), g in (
        ind_ym.groupby(
            [
                "업종_대분류",
                "월"
            ]
        )
    ):

        g = (
            g.set_index(
                "연도"
            )
        )


        # 3개년 모두 있어야 함
        if not all(
            y in g.index
            for y in years
        ):

            continue


        # 각 연도 표본 최소 50개
        if (
            g.loc[
                years,
                "기업수"
            ].min()
            < INDUSTRY_MIN_N
        ):

            continue


        # ----------------------------------------------------
        # 2023 / 2024 / 2025 잔여효과
        # ----------------------------------------------------

        logs = np.array(
            [
                g.loc[
                    y,
                    "잔여효과_log"
                ]
                for y in years
            ]
        )


        pcts = (
            np.expm1(
                logs
            )
            * 100
        )


        # ----------------------------------------------------
        # 3개년 동일가중 대표효과
        # ----------------------------------------------------

        rep_log = np.median(
            logs
        )


        rep_pct = (
            np.expm1(
                rep_log
            )
            * 100
        )


        # ----------------------------------------------------
        # 방향 반복성
        # ----------------------------------------------------

        all_up = np.all(
            logs > 0
        )

        all_down = np.all(
            logs < 0
        )

        same_direction = (
            all_up
            or all_down
        )


        if all_up:

            direction = "상승"

        elif all_down:

            direction = "하락"

        else:

            direction = "불일치"


        # ====================================================
        # Leave-One-Year-Out
        # ====================================================

        before_abs = []

        after_abs = []

        improved = 0


        for test_year in years:

            train_years = [
                y
                for y in years
                if y != test_year
            ]


            train_log = np.median(
                [
                    g.loc[
                        y,
                        "잔여효과_log"
                    ]
                    for y in train_years
                ]
            )


            test_log = (
                g.loc[
                    test_year,
                    "잔여효과_log"
                ]
            )


            before = abs(
                np.expm1(
                    test_log
                )
                * 100
            )


            after = abs(
                np.expm1(
                    test_log
                    - train_log
                )
                * 100
            )


            before_abs.append(
                before
            )


            after_abs.append(
                after
            )


            if after < before:

                improved += 1


        before_mean = np.mean(
            before_abs
        )

        after_mean = np.mean(
            after_abs
        )


        if before_mean > 0:

            reduction = (
                1
                - after_mean
                / before_mean
            ) * 100

        else:

            reduction = np.nan


        # ====================================================
        # 업종별 추가효과 등급
        # ====================================================

        abs_effect = abs(
            rep_pct
        )


        if (
            same_direction
            and abs_effect >= 1
            and improved >= 2
            and reduction >= 20
        ):

            grade = (
                "A_업종추가조정후보"
            )


        elif (
            same_direction
            and abs_effect >= 1
            and improved >= 2
            and reduction > 0
        ):

            grade = (
                "B_업종약한후보"
            )


        elif same_direction:

            grade = (
                "C_반복하지만불안정"
            )


        else:

            grade = "D"


        industry_rows.append(
            {
                "변수":
                    metric_name,

                "업종":
                    industry,

                "월":
                    month,

                "공통A월여부":
                    month
                    in common_A_months,

                "2023_잔여_pct":
                    pcts[0],

                "2024_잔여_pct":
                    pcts[1],

                "2025_잔여_pct":
                    pcts[2],

                "대표잔여효과_pct":
                    rep_pct,

                "방향":
                    direction,

                "최소연도기업수":
                    int(
                        g.loc[
                            years,
                            "기업수"
                        ].min()
                    ),

                "LOO_개선연도수":
                    improved,

                "LOO_감소율_pct":
                    reduction,

                "추가지수후보":
                    np.exp(
                        rep_log
                    ),

                "등급":
                    grade
            }
        )


industry_season_screen = (
    pd.DataFrame(
        industry_rows
    )
)


# ============================================================
# A/B만 출력
# ============================================================

industry_candidates = (
    industry_season_screen[
        industry_season_screen[
            "등급"
        ].isin(
            [
                "A_업종추가조정후보",
                "B_업종약한후보"
            ]
        )
    ]
    .sort_values(
        [
            "등급",
            "변수",
            "업종",
            "월"
        ]
    )
)


print(
    "=" * 130
)

print(
    "업종별 추가 계절성 A/B 후보"
)

print(
    "=" * 130
)


if len(
    industry_candidates
):

    print(
        industry_candidates[
            [
                "변수",
                "업종",
                "월",
                "공통A월여부",
                "2023_잔여_pct",
                "2024_잔여_pct",
                "2025_잔여_pct",
                "대표잔여효과_pct",
                "LOO_개선연도수",
                "LOO_감소율_pct",
                "추가지수후보",
                "등급"
            ]
        ]
        .round(3)
        .to_string(
            index=False
        )
    )

else:

    print(
        "A/B 업종별 추가계절성 후보 없음"
    )

업종별 추가 계절성 A/B 후보
      변수                       업종  월  공통A월여부  2023_잔여_pct  2024_잔여_pct  2025_잔여_pct  대표잔여효과_pct  LOO_개선연도수  LOO_감소율_pct  추가지수후보         등급
  거래금액합계                      건설업  1    True       12.221        4.944       12.965      12.221          2       53.681   1.122 A_업종추가조정후보
  거래금액합계                      건설업  2    True       -7.976       -3.270      -13.450      -7.976          2       30.243   0.920 A_업종추가조정후보
  거래금액합계                      건설업  5   False       -1.320       -3.899       -5.695      -3.899          2       34.064   0.961 A_업종추가조정후보
  거래금액합계                      건설업  8    True       -5.434       -3.517       -3.579      -3.579          3       68.556   0.964 A_업종추가조정후보
  거래금액합계                      건설업 12   False       14.192       16.757       22.069      16.757          3       78.928   1.168 A_업종추가조정후보
  거래금액합계                  교육 서비스업  1    True       -3.111       -5.441       -1.813      -3.111          2       40.321   0.969 A_업종추가조정후보
  거래금액합계 

In [9]:
# ============================================================
# 09-C. 7개 변수 실제 사용도 확인
# ============================================================

usage_rows = []

for metric_name, value_col in METRICS.items():

    s = work[value_col]

    observed = s.notna()

    n_obs = observed.sum()
    n_zero = (s[observed] == 0).sum()
    n_positive = (s[observed] > 0).sum()

    # 기업별로 36개월 중 양수였던 개월 수
    tmp = work.loc[
        observed,
        ["법인ID", value_col]
    ].copy()

    tmp["양수"] = (
        tmp[value_col] > 0
    )

    firm_positive = (
        tmp
        .groupby("법인ID")["양수"]
        .sum()
    )

    usage_rows.append({
        "변수": metric_name,
        "관측건수": n_obs,
        "0건수": n_zero,
        "0비율_pct": n_zero / n_obs * 100,
        "양수건수": n_positive,

        "한번이라도사용기업수":
            (firm_positive >= 1).sum(),

        "6개월이상사용기업수":
            (firm_positive >= 6).sum(),

        "12개월이상사용기업수":
            (firm_positive >= 12).sum(),

        "24개월이상사용기업수":
            (firm_positive >= 24).sum()
    })


usage_summary = pd.DataFrame(
    usage_rows
)

print("=" * 100)
print("7개 변수 사용도 / 0 비율")
print("=" * 100)

print(
    usage_summary
    .round(2)
    .to_string(index=False)
)

7개 변수 사용도 / 0 비율
      변수   관측건수    0건수  0비율_pct   양수건수  한번이라도사용기업수  6개월이상사용기업수  12개월이상사용기업수  24개월이상사용기업수
  예금잔액합계 365988  39901    10.90 326087       14824       12834        10951         7449
  투자상품잔액 365988 286935    78.40  79053        3266        2903         2554         1915
운전자금대출잔액 365988  72988    19.94 293000       13095       11429         9735         6740
시설자금대출잔액 365988 252540    69.00 113448        4892        4359         3858         2596
   카드사용액 365988 135589    37.05 230399       10299        8890         7700         5474
  거래금액합계 365988  12416     3.39 353572       15202       13435        11638         8274
 요구불출금금액 365988  12649     3.46 353339       15190       13427        11630         8271


In [10]:
# ============================================================
# 09-D. 실사용 기업 기준 공통 계절성 재검증
#
# 기업×연도에서
# 해당 변수 > 0인 월이 최소 6개월인 경우만 분석
# ============================================================

active_results = []


for metric_name, value_col in METRICS.items():

    x = work[
        [
            "법인ID",
            "기준월",
            value_col
        ]
    ].copy()

    x = x[
        x[value_col].notna()
        &
        (x[value_col] >= 0)
    ].copy()

    x["연도"] = (
        x["기준월"].dt.year
    )

    x["월"] = (
        x["기준월"].dt.month
    )


    # --------------------------------------------------------
    # 기업×연도 실제 사용개월
    # --------------------------------------------------------

    active_fy = (
        x
        .assign(
            양수=lambda z:
                z[value_col] > 0
        )
        .groupby(
            ["법인ID", "연도"]
        )
        .agg(
            양수월수=(
                "양수",
                "sum"
            )
        )
        .reset_index()
    )

    active_fy = active_fy[
        active_fy["양수월수"] >= 6
    ][
        ["법인ID", "연도"]
    ]


    x = x.merge(
        active_fy,
        on=[
            "법인ID",
            "연도"
        ],
        how="inner"
    )


    # --------------------------------------------------------
    # 기존 함수와 동일한 입력형태의 임시 데이터
    # --------------------------------------------------------

    temp = x[
        [
            "법인ID",
            "기준월",
            value_col
        ]
    ].copy()

    # 함수에 필요한 업종컬럼 연결
    temp = temp.merge(
        work[
            [
                "법인ID",
                "업종_대분류"
            ]
        ]
        .drop_duplicates(
            "법인ID"
        ),
        on="법인ID",
        how="left"
    )


    result, _, _ = (
        analyze_common_seasonality(
            temp,
            metric_name,
            value_col
        )
    )

    active_results.append(
        result
    )


active_season_screen = pd.concat(
    active_results,
    ignore_index=True
)


# ============================================================
# 변수별 A/B/C 요약
# ============================================================

active_overview = []

for metric, g in (
    active_season_screen
    .groupby("변수")
):

    active_overview.append({
        "변수": metric,

        "A_조정후보":
            months_of_grade(
                g,
                "A_조정후보"
            ),

        "B_약한후보":
            months_of_grade(
                g,
                "B_약한후보"
            ),

        "C_불안정":
            months_of_grade(
                g,
                "C_반복하지만불안정"
            )
    })


active_overview = pd.DataFrame(
    active_overview
)


print("\n" + "=" * 110)
print("실사용 기업 기준 계절성 요약")
print("=" * 110)

print(
    active_overview
    .to_string(index=False)
)


print("\n" + "=" * 110)
print("실사용 기업 기준 A등급 상세")
print("=" * 110)

print(
    active_season_screen[
        active_season_screen["등급"]
        == "A_조정후보"
    ][
        [
            "변수",
            "월",
            "2023_pct",
            "2024_pct",
            "2025_pct",
            "대표효과_pct",
            "LOO_개선연도수",
            "LOO_감소율_pct",
            "계절지수후보"
        ]
    ]
    .round(3)
    .to_string(index=False)
)


실사용 기업 기준 계절성 요약
      변수                    A_조정후보 B_약한후보      C_불안정
  거래금액합계      1월, 2월, 8월, 10월, 11월      - 3월, 6월, 9월
시설자금대출잔액                         -      -          -
  예금잔액합계 1월, 2월, 4월, 10월, 11월, 12월      - 7월, 8월, 9월
 요구불출금금액      1월, 2월, 8월, 10월, 11월      -     3월, 6월
운전자금대출잔액                         -      -          -
   카드사용액                        7월      -         9월
  투자상품잔액                       12월      -         4월

실사용 기업 기준 A등급 상세
     변수  월  2023_pct  2024_pct  2025_pct  대표효과_pct  LOO_개선연도수  LOO_감소율_pct  계절지수후보
 예금잔액합계  1    -0.392    -1.453    -2.290    -1.453          2       27.547   0.985
 예금잔액합계  2    -1.633    -0.937    -1.915    -1.633          3       62.134   0.984
 예금잔액합계  4    -0.387    -1.283    -1.837    -1.283          2       32.404   0.987
 예금잔액합계 10    -5.127    -4.651    -2.683    -4.651          3       63.099   0.953
 예금잔액합계 11    -4.087    -2.895    -1.920    -2.895          3       61.046   0.971
 예금잔액합계 12     3.386     0.983     1.840  

In [11]:
# ============================================================
# 09-E. 확정 후보 지수 최종 적용 검증
#
# 대상
# 거래금액합계 : 1, 2, 8, 10, 11월
# 요구불출금금액 : 1, 2, 8, 10, 11월
# 예금잔액합계 : 10, 11월
# 카드사용액 : 7월
#
# 실사용 기업 기준으로 산정했던 지수를
# 동일한 실사용 기업 표본에 적용하여
# 조정 전후 월효과를 최종 확인
# ============================================================

import pandas as pd
import numpy as np


SELECTED_MONTHS = {
    "거래금액합계": [1, 2, 8, 10, 11],
    "요구불출금금액": [1, 2, 8, 10, 11],
    "예금잔액합계": [10, 11],
    "카드사용액": [7]
}


# ------------------------------------------------------------
# 1. 09-D 결과에서 정밀 계절지수 가져오기
# ------------------------------------------------------------

final_index_rows = []

for metric, months in SELECTED_MONTHS.items():

    temp = active_season_screen[
        (active_season_screen["변수"] == metric)
        &
        (active_season_screen["월"].isin(months))
    ].copy()

    for _, row in temp.iterrows():

        final_index_rows.append({
            "변수": metric,
            "월": int(row["월"]),
            "계절지수": float(row["계절지수후보"]),
            "기존대표효과_pct": float(row["대표효과_pct"]),
            "기존_LOO개선연도수": int(row["LOO_개선연도수"]),
            "기존_LOO감소율_pct": float(row["LOO_감소율_pct"])
        })


final_indices = pd.DataFrame(final_index_rows)


print("=" * 100)
print("최종 검증 대상 계절지수")
print("=" * 100)

print(
    final_indices
    .round(5)
    .to_string(index=False)
)


# ============================================================
# 2. 변수별 최종 적용 검증
# ============================================================

validation_rows = []

for metric, months in SELECTED_MONTHS.items():

    value_col = METRICS[metric]

    # --------------------------------------------------------
    # 원자료
    # --------------------------------------------------------

    x = work[
        [
            "법인ID",
            "기준월",
            value_col
        ]
    ].copy()

    x = x[
        x[value_col].notna()
        &
        (x[value_col] >= 0)
    ].copy()

    x["연도"] = x["기준월"].dt.year
    x["월"] = x["기준월"].dt.month


    # --------------------------------------------------------
    # 실사용 기업×연도
    # 양수 월 최소 6개월
    # --------------------------------------------------------

    active_fy = (
        x
        .assign(
            양수=x[value_col] > 0
        )
        .groupby(
            ["법인ID", "연도"]
        )
        .agg(
            양수월수=("양수", "sum")
        )
        .reset_index()
    )

    active_fy = active_fy[
        active_fy["양수월수"] >= 6
    ][
        ["법인ID", "연도"]
    ]

    x = x.merge(
        active_fy,
        on=["법인ID", "연도"],
        how="inner"
    )


    # --------------------------------------------------------
    # 조정 전 금액
    # --------------------------------------------------------

    x["조정전값"] = x[value_col]


    # --------------------------------------------------------
    # 최종 지수 적용
    # --------------------------------------------------------

    index_map = (
        final_indices[
            final_indices["변수"] == metric
        ]
        .set_index("월")["계절지수"]
        .to_dict()
    )

    x["계절지수"] = (
        x["월"]
        .map(index_map)
        .fillna(1.0)
    )

    x["조정후값"] = (
        x["조정전값"]
        / x["계절지수"]
    )


    # ========================================================
    # 3. 조정 전/후 월효과 재계산
    # ========================================================

    for stage, col in [
        ("조정전", "조정전값"),
        ("조정후", "조정후값")
    ]:

        z = x[
            [
                "법인ID",
                "연도",
                "월",
                col
            ]
        ].copy()

        z["log값"] = np.log1p(
            z[col]
        )


        # 기업×연도 수준
        fy = (
            z
            .groupby(
                ["법인ID", "연도"]
            )
            .agg(
                관측월수=("월", "nunique"),
                연간_log평균=("log값", "mean")
            )
            .reset_index()
        )

        fy = fy[
            fy["관측월수"] >= 6
        ]


        z = z.merge(
            fy[
                [
                    "법인ID",
                    "연도",
                    "연간_log평균"
                ]
            ],
            on=["법인ID", "연도"],
            how="inner"
        )


        z["월효과_log"] = (
            z["log값"]
            - z["연간_log평균"]
        )


        ym = (
            z
            .groupby(
                ["연도", "월"]
            )["월효과_log"]
            .median()
            .reset_index()
        )


        ym["효과_pct"] = (
            np.expm1(
                ym["월효과_log"]
            ) * 100
        )


        # 선택된 월만
        ym = ym[
            ym["월"].isin(months)
        ]


        for month in months:

            t = ym[
                ym["월"] == month
            ]["효과_pct"]

            if len(t) != 3:
                continue

            validation_rows.append({
                "변수": metric,
                "월": month,
                "단계": stage,
                "2023_pct": float(
                    ym[
                        (ym["월"] == month)
                        &
                        (ym["연도"] == 2023)
                    ]["효과_pct"].iloc[0]
                ),
                "2024_pct": float(
                    ym[
                        (ym["월"] == month)
                        &
                        (ym["연도"] == 2024)
                    ]["효과_pct"].iloc[0]
                ),
                "2025_pct": float(
                    ym[
                        (ym["월"] == month)
                        &
                        (ym["연도"] == 2025)
                    ]["효과_pct"].iloc[0]
                ),
                "평균절대효과": float(
                    np.mean(
                        np.abs(t)
                    )
                )
            })


validation_long = pd.DataFrame(
    validation_rows
)


# ============================================================
# 4. 조정 전후 결합
# ============================================================

before = (
    validation_long[
        validation_long["단계"] == "조정전"
    ]
    .drop(columns="단계")
    .rename(
        columns={
            "2023_pct": "전_2023",
            "2024_pct": "전_2024",
            "2025_pct": "전_2025",
            "평균절대효과": "전_평균절대효과"
        }
    )
)


after = (
    validation_long[
        validation_long["단계"] == "조정후"
    ]
    .drop(columns="단계")
    .rename(
        columns={
            "2023_pct": "후_2023",
            "2024_pct": "후_2024",
            "2025_pct": "후_2025",
            "평균절대효과": "후_평균절대효과"
        }
    )
)


final_validation = before.merge(
    after,
    on=["변수", "월"],
    how="inner"
)


final_validation["최종감소율_pct"] = (
    1
    - (
        final_validation[
            "후_평균절대효과"
        ]
        /
        final_validation[
            "전_평균절대효과"
        ]
    )
) * 100


print("\n" + "=" * 130)
print("최종 계절지수 적용 전후 검증")
print("=" * 130)

print(
    final_validation[
        [
            "변수",
            "월",
            "전_2023",
            "전_2024",
            "전_2025",
            "후_2023",
            "후_2024",
            "후_2025",
            "전_평균절대효과",
            "후_평균절대효과",
            "최종감소율_pct"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

최종 검증 대상 계절지수
     변수  월    계절지수  기존대표효과_pct  기존_LOO개선연도수  기존_LOO감소율_pct
 거래금액합계  1 1.03551     3.55142            3       58.91529
 거래금액합계  2 0.95938    -4.06169            2       39.07953
 거래금액합계  8 0.91738    -8.26219            2       52.12812
 거래금액합계 10 0.95767    -4.23308            3       59.67235
 거래금액합계 11 0.89606   -10.39444            2       41.84640
요구불출금금액  1 1.05084     5.08434            3       72.66191
요구불출금금액  2 0.96027    -3.97259            2       47.26844
요구불출금금액  8 0.92697    -7.30312            2       44.74137
요구불출금금액 10 0.97057    -2.94278            2       33.21832
요구불출금금액 11 0.89951   -10.04895            2       40.60657
 예금잔액합계 10 0.95349    -4.65107            3       63.09898
 예금잔액합계 11 0.97105    -2.89468            3       61.04578
  카드사용액  7 1.05340     5.33971            3       63.45092

최종 계절지수 적용 전후 검증
     변수  월  전_2023  전_2024  전_2025  후_2023  후_2024  후_2025  전_평균절대효과  후_평균절대효과  최종감소율_pct
 거래금액합계  1   3.551   6.074   3.128  -0.822   0.993  